# 🧠 LasmoidV1 — 100M Parameter Training on Kaggle GPU T4 x2

**Architecture:** MLA + mHC + MoE + CQRS + MTP  
**Target:** ~100M parameters, gpt2 tokenizer, 512 seq len  
**Datasets:** openbmb/Ultra-FineWeb-L3 (50%) + WithinUsAI/claude_mythos_distilled_25k (35%) + HelioAI/Claude-Opus-4.8-DeepThink-462x-105M (15%)  
**Checkpoints:** Auto-synced to Hugging Face Hub every 500 steps — survives Kaggle session timeouts!

### Prerequisites
Add `HF_TOKEN` as a Kaggle Secret (Settings → Secrets → Add New Secret).  
Name it exactly: **`HF_TOKEN`**

In [ ]:
# ── CELL 1: Environment diagnostics ───────────────────────────────────
import subprocess, sys, os

print('=== GPU Info ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found!')

print('=== Python ===')
print(sys.version)

print('=== Disk Space ===')
subprocess.run(['df', '-h', '/kaggle/working'], text=True)

print('=== RAM ===')
with open('/proc/meminfo') as f:
    for line in f:
        if 'MemTotal' in line or 'MemAvailable' in line:
            print(line.strip())

In [ ]:
# ── CELL 2: Install dependencies ───────────────────────────────────────
!pip install -q tiktoken datasets transformers huggingface_hub accelerate
print('All dependencies installed ✓')

In [ ]:
# ── CELL 3: Clone repo from GitHub ─────────────────────────────────────
# Option A: Clone from GitHub (if your repo is public or you have a token)
# Replace with your actual GitHub repo URL
import os

REPO_URL = 'https://github.com/Theory903/Lasmoid-V1.git'
WORK_DIR = '/kaggle/working/Lasmoid-V1'

if not os.path.exists(WORK_DIR):
    result = os.system(f'git clone {REPO_URL} {WORK_DIR}')
    if result != 0:
        print('Git clone failed. You may need to upload files manually (see Cell 3b).')
    else:
        print(f'Cloned repo to {WORK_DIR} ✓')
else:
    print(f'Repo already exists at {WORK_DIR}')
    os.system(f'cd {WORK_DIR} && git pull')

In [ ]:
# ── CELL 3b: ALTERNATIVE — Upload files manually ───────────────────────
# If you don't have a public GitHub repo, use Kaggle's "Add Input" → "Upload"
# to add a ZIP of your project, then:
#
# import shutil
# shutil.unpack_archive('/kaggle/input/lasmoid-v1/Lasmoid-V1.zip', '/kaggle/working/Lasmoid-V1')
#
# OR write the code inline — see Cell 4 which writes model.py directly.
print('If you used git clone in Cell 3, skip this cell.')

In [ ]:
# ── CELL 4: Hugging Face Authentication via Kaggle Secret ──────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')

login(token=HF_TOKEN)
api = HfApi()
username = api.whoami(token=HF_TOKEN)['name']
print(f'Logged in as: {username} ✓')

# Set as environment variable so train_kaggle.py picks it up
os.environ['HF_TOKEN'] = HF_TOKEN
print(f'HF_TOKEN set. Checkpoints will sync to: {username}/lasmoid-100m')

In [ ]:
# ── CELL 5: Verify model can initialize ───────────────────────────────
import sys
WORK_DIR = '/kaggle/working/Lasmoid-V1'
sys.path.insert(0, WORK_DIR)

import torch
from inference.model import LasmoidV1, ModelArgs

model_args = ModelArgs(
    dim=512,
    n_layers=8,
    n_heads=8,
    head_dim=64,
    q_lora_rank=64,
    o_lora_rank=64,
    n_routed_experts=4,
    n_shared_experts=1,
    n_activated_experts=2,
    moe_inter_dim=1024,
    max_seq_len=512,
    max_batch_size=8,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = LasmoidV1(model_args).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Device: {device}')
print(f'Model: LasmoidV1, {total_params:,} parameters')
print('Model initialized successfully ✓')

# Quick forward pass sanity check
x = torch.randint(0, 50257, (2, 64)).to(device)
with torch.no_grad():
    out = model(x, x)
print(f'Forward pass output shape: {out[0].shape}')
print('Sanity check passed ✓')

In [ ]:
# ── CELL 6: LAUNCH TRAINING ───────────────────────────────────────────
# 
# Configuration:
#   --max_iters 50000   → Full training run (saves every 500 steps to HF Hub)
#   --batch_size 8      → 8 sequences per microbatch  
#   --grad_accum 4      → Effective batch = 32 sequences = 32*512 = 16K tokens/step
#   --learning_rate 6e-4
#   --warmup_steps 1000 → Gradual LR ramp
#
# Kaggle T4 x2: ~25 min/1000 steps → 50000 steps ≈ 20 hours
# (Session limit is 12h, but HF Hub auto-resume means you can restart!)
#
import subprocess, os

WORK_DIR = '/kaggle/working/Lasmoid-V1'
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)

cmd = [
    'python', f'{WORK_DIR}/train_kaggle.py',
    '--max_iters', '50000',
    '--batch_size', '8',
    '--grad_accum', '4',
    '--learning_rate', '6e-4',
    '--warmup_steps', '1000',
    '--save_interval', '500',
    '--checkpoint_dir', f'{WORK_DIR}/checkpoints',
    '--session_hours', '11',
]

print('Starting LasmoidV1 training...')
print(f'Command: {" ".join(cmd)}')
print('=' * 70)

# Run with live output streaming
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=WORK_DIR
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
print(f'\nTraining finished with return code: {process.returncode}')

In [ ]:
# ── CELL 7: Verify checkpoint on HF Hub ───────────────────────────────
# Run this anytime to see what's saved remotely
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
api = HfApi()
username = api.whoami(token=HF_TOKEN)['name']
repo_id = f'{username}/lasmoid-100m'

try:
    files = list(api.list_repo_files(repo_id=repo_id))
    pt_files = [f for f in files if f.endswith('.pt')]
    print(f'Repository: {repo_id}')
    print(f'Checkpoints found ({len(pt_files)}):')
    for f in sorted(pt_files):
        print(f'  - {f}')
    if not pt_files:
        print('No checkpoints yet — training may still be in progress.')
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── CELL 8: Quick generation test after training ───────────────────────
import sys, torch, tiktoken
WORK_DIR = '/kaggle/working/Lasmoid-V1'
sys.path.insert(0, WORK_DIR)

from inference.model import LasmoidV1, ModelArgs

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = tiktoken.get_encoding('gpt2')

model_args = ModelArgs(
    dim=512, n_layers=8, n_heads=8, head_dim=64,
    q_lora_rank=64, o_lora_rank=64,
    n_routed_experts=4, n_shared_experts=1, n_activated_experts=2,
    moe_inter_dim=1024, max_seq_len=512, max_batch_size=1,
)

# Load latest checkpoint
import glob
ckpts = sorted(glob.glob(f'{WORK_DIR}/checkpoints/lasmoid_checkpoint_step_*.pt'))
if not ckpts:
    ckpts = glob.glob(f'{WORK_DIR}/checkpoints/lasmoid_latest.pt')

if ckpts:
    latest = ckpts[-1]
    print(f'Loading: {latest}')
    model = LasmoidV1(model_args).to(device)
    ckpt = torch.load(latest, map_location=device)
    model.load_state_dict(ckpt.get('model_state_dict', ckpt))
    model.eval()

    prompt = 'The meaning of intelligence is'
    tokens = tokenizer.encode(prompt)
    x = torch.tensor([tokens], dtype=torch.long).to(device)

    print(f'Prompt: {prompt}')
    print('Generation:', end=' ')
    with torch.no_grad():
        for _ in range(50):
            logits, _, _, _ = model(x, x)
            next_tok = logits[0, -1].argmax().item()
            print(tokenizer.decode([next_tok]), end='', flush=True)
            x = torch.cat([x, torch.tensor([[next_tok]]).to(device)], dim=1)
            if x.shape[1] >= 512:
                break
    print()
else:
    print('No checkpoint found. Run Cell 6 first.')